# 08 — Reddit EDA: r/Forex · r/investing · r/stocks · r/Economics

**Goal:** Understand the structure, quality, and signal potential of the collected Reddit data across four communities.

**Approach:** Run each cell, observe the output, write conclusions in the Observations blocks below each result. No assumptions are made in advance.

## 1. Setup

In [ ]:
import json
import warnings
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

ROOT    = Path().resolve().parent
RAW_DIR = ROOT / "data" / "raw" / "reddit"

FILES = {
    "Forex"    : RAW_DIR / "forex_raw_20260405_192900.jsonl",
    "investing": RAW_DIR / "investing_raw_20260405_210126.jsonl",
    "stocks"   : RAW_DIR / "stocks_raw_20260405_210126.jsonl",
    "Economics": RAW_DIR / "economics_raw_20260405_210126.jsonl",
}

# Only load the columns we need — keeps memory manageable on large JSONL files
KEEP = {
    "id", "subreddit", "author", "created_utc",
    "title", "selftext",
    "score", "upvote_ratio", "num_comments",
    "is_video", "is_self", "is_gallery", "is_reddit_media_domain",
    "domain", "url", "link_flair_text",
    "removed_by_category",
}

for label, path in FILES.items():
    exists = path.exists()
    size   = f"{path.stat().st_size / 1_048_576:.0f} MB" if exists else "MISSING"
    print(f"  r/{label:<12}  {size:<10}  {path.name}")

## 2. Load Data

In [ ]:
def load_jsonl(path: Path, keep: set[str]) -> pd.DataFrame:
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            raw = json.loads(line)
            records.append({k: raw.get(k) for k in keep})
    return pd.DataFrame(records)

frames = {}
for label, path in FILES.items():
    print(f"Loading r/{label}...", end=" ", flush=True)
    df = load_jsonl(path, KEEP)
    frames[label] = df
    print(f"{len(df):,} posts  |  {df.memory_usage(deep=True).sum() / 1_048_576:.0f} MB in RAM")

df_all = pd.concat(frames.values(), ignore_index=True)

# Derived columns
df_all["created_dt"]   = pd.to_datetime(df_all["created_utc"], unit="s", utc=True)
df_all["year"]         = df_all["created_dt"].dt.year
df_all["month"]        = df_all["created_dt"].dt.to_period("M")
df_all["hour"]         = df_all["created_dt"].dt.hour
df_all["weekday"]      = df_all["created_dt"].dt.day_name()
df_all["text_length"]  = (df_all["title"].fillna("") + " " + df_all["selftext"].fillna("")).str.len()
df_all["is_deleted"]   = df_all["removed_by_category"].isin(["deleted", "moderator", "author"])

def classify_type(row):
    if row.get("is_video"):               return "video"
    if row.get("is_gallery"):             return "gallery"
    if row.get("is_self"):                return "text"
    if row.get("is_reddit_media_domain"): return "image"
    return "external"

df_all["post_type"] = df_all.apply(classify_type, axis=1)

print(f"\nCombined: {len(df_all):,} posts  |  {df_all.memory_usage(deep=True).sum() / 1_048_576:.0f} MB")

## 3. Dataset Overview

In [ ]:
# Per-subreddit summary
summary = df_all.groupby("subreddit").agg(
    posts        = ("id",           "count"),
    unique_authors=("author",       "nunique"),
    date_min     = ("created_dt",   "min"),
    date_max     = ("created_dt",   "max"),
    deleted_pct  = ("is_deleted",   lambda x: f"{x.mean()*100:.1f}%"),
    median_score = ("score",        "median"),
    median_comments=("num_comments","median"),
).reset_index()

summary["date_min"] = summary["date_min"].dt.strftime("%Y-%m-%d")
summary["date_max"] = summary["date_max"].dt.strftime("%Y-%m-%d")
print(summary.to_string(index=False))

**Observations — Dataset overview:**
*(fill in after running)*

## 4. Data Quality

In [ ]:
# Deleted / removed breakdown per subreddit
del_breakdown = df_all.groupby(["subreddit", "removed_by_category"]).size().unstack(fill_value=0)
del_breakdown["TOTAL"] = del_breakdown.sum(axis=1)
print("Removed-by category counts per subreddit:")
print(del_breakdown.to_string())

In [ ]:
# What fraction of posts have usable text?
df_all["has_body"] = df_all["selftext"].fillna("").str.strip().apply(
    lambda x: x not in ("", "[deleted]", "[removed]")
)

text_quality = df_all.groupby("subreddit").agg(
    total     = ("id",       "count"),
    has_body  = ("has_body", "sum"),
    title_only= ("has_body", lambda x: (~x).sum()),
).assign(
    body_pct = lambda d: (d["has_body"] / d["total"] * 100).round(1)
)
print(text_quality.to_string())

**Observations — Data quality:**
*(fill in after running)*

## 5. Post Type Distribution

In [ ]:
type_counts = (
    df_all.groupby(["subreddit", "post_type"])
    .size()
    .unstack(fill_value=0)
)
type_pct = type_counts.div(type_counts.sum(axis=1), axis=0) * 100
print("Post type distribution (%) per subreddit:")
print(type_pct.round(1).to_string())

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5), sharey=False)
colors = sns.color_palette("muted", n_colors=len(type_pct.columns))

for ax, (sr, row) in zip(axes, type_pct.iterrows()):
    row_sorted = row.sort_values(ascending=False)
    bars = ax.bar(row_sorted.index, row_sorted.values, color=colors[:len(row_sorted)])
    ax.set_title(f"r/{sr}", fontsize=12, fontweight="bold")
    ax.set_ylabel("% of posts")
    ax.set_ylim(0, 100)
    for bar in bars:
        h = bar.get_height()
        if h > 2:
            ax.text(bar.get_x() + bar.get_width()/2, h + 1, f"{h:.1f}%", ha="center", fontsize=9)

fig.suptitle("Post Type Distribution by Subreddit", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

**Observations — Post types:**
*(fill in after running)*

## 6. Temporal Analysis

In [ ]:
# Monthly post volume per subreddit
monthly = (
    df_all.groupby(["subreddit", df_all["created_dt"].dt.to_period("M")])
    .size()
    .reset_index(name="posts")
)
monthly["created_dt"] = monthly["created_dt"].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(16, 5))
for sr, grp in monthly.groupby("subreddit"):
    ax.plot(grp["created_dt"], grp["posts"], label=f"r/{sr}", linewidth=1.8)

ax.set_title("Monthly Post Volume (2021–2025)", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Posts per month")
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

In [ ]:
# Yearly growth rates
yearly = df_all.groupby(["subreddit", "year"]).size().unstack(fill_value=0)
print("Posts per year per subreddit:")
print(yearly.to_string())

print("\nYear-over-year growth (%):")
print(yearly.pct_change(axis=1).mul(100).round(1).to_string())

In [ ]:
# Posting by hour of day (UTC)
hour_dist = df_all.groupby(["subreddit", "hour"]).size().unstack(fill_value=0)
hour_pct  = hour_dist.div(hour_dist.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(14, 4))
for sr, row in hour_pct.iterrows():
    ax.plot(row.index, row.values, label=f"r/{sr}", linewidth=1.8, marker="o", markersize=3)

ax.set_title("Post Volume by Hour of Day (UTC)", fontsize=13, fontweight="bold")
ax.set_xlabel("Hour (UTC)")
ax.set_ylabel("% of posts")
ax.set_xticks(range(0, 24))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Posting by weekday
day_order  = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
day_dist   = df_all.groupby(["subreddit", "weekday"]).size().unstack(fill_value=0)
day_dist   = day_dist.reindex(columns=day_order, fill_value=0)
day_pct    = day_dist.div(day_dist.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(12, 4))
for sr, row in day_pct.iterrows():
    ax.plot(row.index, row.values, label=f"r/{sr}", linewidth=1.8, marker="o", markersize=4)

ax.set_title("Post Volume by Day of Week", fontsize=13, fontweight="bold")
ax.set_ylabel("% of posts")
ax.legend()
plt.tight_layout()
plt.show()

**Observations — Temporal patterns:**
*(fill in after running)*

## 7. Engagement Analysis

In [ ]:
# Score distribution — use log scale due to power-law
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=False)
for ax, (sr, grp) in zip(axes, df_all.groupby("subreddit")):
    scores = grp["score"].clip(lower=1)
    ax.hist(np.log10(scores), bins=50, color=sns.color_palette("muted")[0], edgecolor="none")
    ax.set_title(f"r/{sr}", fontsize=11, fontweight="bold")
    ax.set_xlabel("log10(score)")
    ax.set_ylabel("Posts")
    p50 = scores.median()
    p95 = scores.quantile(0.95)
    ax.axvline(np.log10(p50), color="red",    linestyle="--", linewidth=1, label=f"p50={p50:.0f}")
    ax.axvline(np.log10(p95), color="orange", linestyle="--", linewidth=1, label=f"p95={p95:.0f}")
    ax.legend(fontsize=8)

fig.suptitle("Score Distribution (log10 scale)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Engagement summary stats
eng = df_all.groupby("subreddit").agg(
    score_median  = ("score",        "median"),
    score_mean    = ("score",        "mean"),
    score_p95     = ("score",        lambda x: x.quantile(0.95)),
    score_max     = ("score",        "max"),
    comments_median=("num_comments", "median"),
    comments_mean = ("num_comments", "mean"),
    upvote_ratio  = ("upvote_ratio", "mean"),
).round(1)
print(eng.to_string())

In [ ]:
# Score vs comments correlation per subreddit
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (sr, grp) in zip(axes, df_all.groupby("subreddit")):
    sample = grp.sample(min(5000, len(grp)), random_state=42)
    ax.scatter(
        np.log1p(sample["score"]),
        np.log1p(sample["num_comments"]),
        alpha=0.15, s=5, color=sns.color_palette("muted")[1]
    )
    corr = grp[["score", "num_comments"]].corr().iloc[0, 1]
    ax.set_title(f"r/{sr}  (r={corr:.2f})", fontsize=10, fontweight="bold")
    ax.set_xlabel("log(score+1)")
    ax.set_ylabel("log(comments+1)")

fig.suptitle("Score vs Comments Correlation", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Engagement by post type
eng_by_type = df_all.groupby(["subreddit", "post_type"]).agg(
    n             = ("id",           "count"),
    score_median  = ("score",        "median"),
    comments_median=("num_comments", "median"),
).round(1)
print(eng_by_type.to_string())

**Observations — Engagement:**
*(fill in after running)*

## 8. Author Analysis

In [ ]:
# Top authors per subreddit (exclude bots & deleted)
bot_list = {"AutoModerator", "[deleted]", "[removed]", None}

for sr, grp in df_all.groupby("subreddit"):
    top = (
        grp[~grp["author"].isin(bot_list)]
        .groupby("author")["id"].count()
        .sort_values(ascending=False)
        .head(10)
    )
    total_authors = grp[~grp["author"].isin(bot_list)]["author"].nunique()
    top10_posts   = top.sum()
    total_posts   = len(grp)
    print(f"\nr/{sr}  ({total_authors:,} unique authors)")
    print(f"  Top 10 authors account for {top10_posts/total_posts*100:.1f}% of posts")
    for author, n in top.items():
        print(f"    {author:<30} {n:>5} posts")

In [ ]:
# Post frequency distribution — how concentrated is posting activity?
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (sr, grp) in zip(axes, df_all.groupby("subreddit")):
    freq = (
        grp[~grp["author"].isin(bot_list)]
        .groupby("author")["id"].count()
    )
    ax.hist(np.log10(freq.clip(lower=1)), bins=40,
            color=sns.color_palette("muted")[2], edgecolor="none")
    ax.set_title(f"r/{sr}", fontsize=11, fontweight="bold")
    ax.set_xlabel("log10(posts per author)")
    ax.set_ylabel("Authors")
    one_post_pct = (freq == 1).mean() * 100
    ax.text(0.98, 0.95, f"{one_post_pct:.0f}% post once",
            transform=ax.transAxes, ha="right", va="top", fontsize=9)

fig.suptitle("Author Post Frequency Distribution", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

**Observations — Authors:**
*(fill in after running)*

## 9. Flair & Topic Distribution

In [ ]:
# Top flairs per subreddit
for sr, grp in df_all.groupby("subreddit"):
    top_flairs = (
        grp["link_flair_text"].dropna()
        .value_counts()
        .head(10)
    )
    total_with_flair = grp["link_flair_text"].notna().sum()
    print(f"\nr/{sr}  ({total_with_flair/len(grp)*100:.1f}% posts have flair)")
    for flair, n in top_flairs.items():
        print(f"    {str(flair):<35} {n:>6,}")

In [ ]:
# Visualise top 8 flairs per subreddit
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (sr, grp) in zip(axes, df_all.groupby("subreddit")):
    top = grp["link_flair_text"].dropna().value_counts().head(8)
    labels = [str(l)[:25] for l in top.index]
    ax.barh(labels[::-1], top.values[::-1], color=sns.color_palette("muted")[3])
    ax.set_title(f"r/{sr}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Posts")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

fig.suptitle("Top 8 Post Flairs by Subreddit", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

**Observations — Flairs / topics:**
*(fill in after running)*

## 10. Content Analysis

In [ ]:
# Text length distribution
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (sr, grp) in zip(axes, df_all.groupby("subreddit")):
    lengths = grp["text_length"].clip(upper=5000)
    ax.hist(lengths, bins=60, color=sns.color_palette("muted")[4], edgecolor="none")
    ax.set_title(f"r/{sr}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Characters (capped 5k)")
    ax.set_ylabel("Posts")
    ax.axvline(lengths.median(), color="red", linestyle="--", linewidth=1,
               label=f"median={lengths.median():.0f}")
    ax.legend(fontsize=8)

fig.suptitle("Post Text Length Distribution", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Top external domains per subreddit
external_posts = df_all[df_all["post_type"] == "external"]
for sr, grp in external_posts.groupby("subreddit"):
    top_domains = grp["domain"].value_counts().head(10)
    print(f"\nr/{sr} — top domains ({len(grp):,} external posts):")
    for dom, n in top_domains.items():
        print(f"    {dom:<40} {n:>6,}")

**Observations — Content:**
*(fill in after running)*

## 11. Cross-Subreddit Comparison

In [ ]:
# Heatmap: normalised engagement metrics across subreddits
metrics = df_all.groupby("subreddit").agg(
    score_median   = ("score",        "median"),
    comments_median= ("num_comments", "median"),
    upvote_ratio   = ("upvote_ratio", "mean"),
    text_length    = ("text_length",  "median"),
    has_body_pct   = ("has_body",     "mean"),
)

metrics_norm = (metrics - metrics.min()) / (metrics.max() - metrics.min())

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(
    metrics_norm.T,
    annot=metrics.T.round(2),
    fmt="g",
    cmap="YlOrRd",
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "Normalised value"},
)
ax.set_title("Cross-Subreddit Engagement Heatmap (raw values annotated)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Monthly activity correlation between subreddits — do they spike together?
monthly_pivot = (
    df_all.groupby(["subreddit", df_all["created_dt"].dt.to_period("M")])
    .size()
    .unstack(level=0)
    .fillna(0)
)

corr = monthly_pivot.corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title("Monthly Post Volume Correlation\n(do subreddits spike together?)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

**Observations — Cross-subreddit comparison:**
*(fill in after running)*

## 12. Signal-Relevant Patterns

Identify features that could carry predictive signal for FX markets.

In [ ]:
# High-engagement post spikes — monthly top-1% score threshold
# These are the posts that "broke through" the community — potentially driven by major events
top1_threshold = df_all.groupby("subreddit")["score"].transform(lambda x: x.quantile(0.99))
viral_posts = df_all[df_all["score"] >= top1_threshold].copy()

print(f"Viral posts (top 1% score per subreddit): {len(viral_posts):,}")

viral_monthly = (
    viral_posts.groupby(["subreddit", viral_posts["created_dt"].dt.to_period("M")])
    .size()
    .unstack(level=0)
    .fillna(0)
)

fig, ax = plt.subplots(figsize=(16, 4))
for col in viral_monthly.columns:
    ax.plot(viral_monthly.index.to_timestamp(), viral_monthly[col], label=f"r/{col}", linewidth=1.5)
ax.set_title("Monthly Count of Viral Posts (top 1% score)", fontsize=13, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Viral posts / month")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Ratio of high-comment to low-comment posts over time (controversy proxy)
# High comments + low score = contested / divisive post
df_all["controversy_proxy"] = (
    df_all["num_comments"] / (df_all["score"].clip(lower=1))
)

controversy_monthly = (
    df_all.groupby(["subreddit", df_all["created_dt"].dt.to_period("M")])["controversy_proxy"]
    .median()
    .unstack(level=0)
)

fig, ax = plt.subplots(figsize=(16, 4))
for col in controversy_monthly.columns:
    ax.plot(controversy_monthly.index.to_timestamp(), controversy_monthly[col],
            label=f"r/{col}", linewidth=1.5)
ax.set_title("Monthly Controversy Proxy (comments / score) — higher = more contested",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Median comments/score")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Weekend vs weekday posting activity ratio
df_all["is_weekend"] = df_all["weekday"].isin(["Saturday", "Sunday"])
weekend_ratio = df_all.groupby("subreddit")["is_weekend"].mean().mul(100).round(1)
print("Weekend posting share (%):")
print(weekend_ratio.to_string())
print("\n(Market is closed weekends — high weekend activity = retail-dominated community)")

In [ ]:
# Engagement by year — is the community becoming more or less active over time?
yearly_eng = df_all.groupby(["subreddit", "year"]).agg(
    score_median   = ("score",        "median"),
    comments_median= ("num_comments", "median"),
).unstack(level=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, metric in zip(axes, ["score_median", "comments_median"]):
    for sr in yearly_eng[metric].columns:
        ax.plot(yearly_eng.index, yearly_eng[metric][sr], label=f"r/{sr}", marker="o")
    ax.set_title(f"Median {metric.replace('_', ' ').title()} by Year",
                 fontsize=11, fontweight="bold")
    ax.set_xlabel("Year")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

**Observations — Signal-relevant patterns:**
*(fill in after running)*

## 13. Summary of Insights

*(Consolidate all observations from sections above into actionable conclusions for signal extraction.)*

| # | Insight | Implication for signal |
|---|---------|------------------------|
| 1 | | |
| 2 | | |
| 3 | | |
| 4 | | |
| 5 | | |